**Kaggle Kernel Config:**

In [ ]:
# KAGGLE_CONFIG: execute = true
# KAGGLE_CONFIG: slug = "notebook-consuming-gpu-p100"
# KAGGLE_CONFIG: language = "python"
# KAGGLE_CONFIG: kernel_type = "notebook"
# KAGGLE_CONFIG: is_private = false
# KAGGLE_CONFIG: enable_gpu = true
# KAGGLE_CONFIG: enable_tpu = false
# KAGGLE_CONFIG: enable_internet = true
# KAGGLE_CONFIG: machine_shape = "NvidiaTeslaP100"
# KAGGLE_CONFIG: dataset_sources = []
# KAGGLE_CONFIG: competition_sources = []
# KAGGLE_CONFIG: kernel_sources = []
# KAGGLE_CONFIG: model_sources = []
# KAGGLE_CONFIG: keywords = ["classification"]

In [1]:
from datetime import datetime, timezone
print('executed at:',datetime.now(timezone.utc).isoformat())

executed at: 2026-08-14T08:58:31.604932+00:00


# Notebook Consuming : GPU - NvidiaTeslaP100 - 16GB

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torch', 'torchvision', 'torchaudio'])
subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', '--no-cache-dir', 'torch==2.10.0', 'torchvision', 'torchaudio', '--index-url', 'https://download.pytorch.org/whl/cu126'])

## Importing Libraries

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

## 1. Automatic GPU Detection

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")

## 2. Data Preparation

In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("Capability:", torch.cuda.get_device_capability(0))
    
    print("Compiled architectures:")
    print(torch.cuda.get_arch_list())

In [ ]:
# Generating a synthetic multi-class classification dataset
X_raw, y_raw = make_classification(
    n_samples=2000, 
    n_features=20, 
    n_classes=3, 
    n_informative=15, 
    random_state=42
)

X_train, X_test, y_train, y_test = train_test_split(
    X_raw, y_raw, test_size=0.2, random_state=42, stratify=y_raw
)

# Convert to PyTorch Tensors
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

## 3.Model Definition

In [ ]:
class DeepClassifier(nn.Module):
    def __init__(self, input_dim, num_classes):
        super(DeepClassifier, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, num_classes)
        )

    def forward(self, x):
        return self.net(x)

model = DeepClassifier(input_dim=20, num_classes=3).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

## 4. Training Loop (GPU Accelerated)

In [ ]:
epochs = 25
print("\nStarting Training...")
model.train()

for epoch in range(epochs):
    running_loss = 0.0
    for inputs, labels in train_loader:
        # Move inputs and labels to the designated device (GPU or CPU)
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)

    epoch_loss = running_loss / len(train_loader.dataset)
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoch {epoch+1}/{epochs} - Loss: {epoch_loss:.4f}")

## 5. Model Evaluation

In [ ]:
model.eval()
with torch.no_grad():
    # Move test dataset to GPU for inference
    X_test_device = X_test_tensor.to(device)
    outputs = model(X_test_device)
    _, preds = torch.max(outputs, 1)

# Bring predictions back to CPU for Scikit-Learn evaluation
y_pred = preds.cpu().numpy()

accuracy = accuracy_score(y_test, y_pred)
print("\n" + "=" * 50)
print(f"Model Accuracy: {accuracy * 100:.2f}%")
print("=" * 50)

print("\n--- Classification Report ---")
print(classification_report(y_test, y_pred))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
print("--- Confusion Matrix ---")
print(cm)

plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.title("Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.tight_layout()
plt.show()

## 6. Save Artifacts to Kaggle Path

In [ ]:
kaggle_output_dir = "/kaggle/working" if os.path.exists("/kaggle") else "./output"
os.makedirs(kaggle_output_dir, exist_ok=True)

model_path = os.path.join(kaggle_output_dir, "deep_learning_model.pth")
torch.save(model.state_dict(), model_path)

print(f"\n Model weights successfully saved to: {model_path}")